# Preparation for Colab

Make sure you're running a GPU runtime; if not, select "GPU" as the hardware accelerator in Runtime > Change Runtime Type in the menu. The next cells will install the `clip` package and its dependencies, and check if PyTorch 1.7.1 or later is installed.

In [1]:
! pip install ftfy regex tqdm
! pip install git+https://github.com/openai/CLIP.git

     |████████████████████████████████| 64 kB 1.3 MB/s 
  Created wheel for ftfy: filename=ftfy-6.0.3-py3-none-any.whl size=41934 sha256=90ec193331444b2c4ff1cd81935e7de42065b89d304db7efac67bcfd87c27873
  Stored in directory: /root/.cache/pip/wheels/19/f5/38/273eb3b5e76dfd850619312f693716ac4518b498f5ffb6f56d
Successfully built ftfy
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-hqnbveqi
  Running command git clone -q https://github.com/openai/CLIP.git /tmp/pip-req-build-hqnbveqi
  Created wheel for clip: filename=clip-1.0-py3-none-any.whl size=1369080 sha256=fda43d2b80cfb2b33c2d43e23ea5f53293a9a8b48d5f9e341de527f6adfbf5a3
  Stored in directory: /tmp/pip-ephem-wheel-cache-kmmplf44/wheels/fd/b9/c3/5b4470e35ed76e174bff77c92f91da82098d5e35fd5bc8cdac
Successfully built clip


In [2]:
# 导入NumPy库，用于数值计算和数组操作
import numpy as np

# 导入PyTorch深度学习框架
import torch

# 导入OpenAI的CLIP库，用于多模态（图像-文本）学习
import clip

# 从tqdm.notebook导入进度条，用于Jupyter Notebook中的循环进度显示
# tqdm: 阿拉伯语"taqaddum"的缩写，意为"进步"
from tqdm.notebook import tqdm

# 导入pkg_resources中的packaging模块，用于版本检查和包管理
from pkg_resources import packaging

# 打印PyTorch版本信息
# 在CLIP项目中，需要特定版本的PyTorch以确保兼容性
# torch.__version__: PyTorch的版本字符串
print("Torch version:", torch.__version__)

Torch version: 1.9.0+cu102


# Loading the model【加载模型】

Download and instantiate a CLIP model using the `clip` module that we just installed.

In [3]:
clip.available_models()

['RN50', 'RN101', 'RN50x4', 'RN50x16', 'ViT-B/32', 'ViT-B/16']

In [4]:
model, preprocess = clip.load("ViT-B/32")

100%|███████████████████████████████████████| 338M/338M [00:05<00:00, 63.6MiB/s]


In [5]:
input_resolution = model.visual.input_resolution
context_length = model.context_length
vocab_size = model.vocab_size

print("Model parameters:", f"{np.sum([int(np.prod(p.shape)) for p in model.parameters()]):,}")
print("Input resolution:", input_resolution)
print("Context length:", context_length)
print("Vocab size:", vocab_size)

Model parameters: 151,277,313
Input resolution: 224
Context length: 77
Vocab size: 49408


# Preparing ImageNet labels and prompts

The following cell contains the 1,000 labels for the ImageNet dataset, followed by the text templates we'll use as "prompt engineering".

以下单元格包含 ImageNet 数据集的 1000 个标签，以及我们将用作“提示设计”的文本模板。

In [6]:
imagenet_classes = ["tench", "goldfish", "great white shark", "tiger shark", "hammerhead shark", "electric ray", "stingray", "rooster", "hen", "ostrich", "brambling", "goldfinch", "house finch", "junco", "indigo bunting", "American robin", "bulbul", "jay", "magpie", "chickadee", "American dipper", "kite (bird of prey)", "bald eagle", "vulture", "great grey owl", "fire salamander", "smooth newt", "newt", "spotted salamander", "axolotl", "American bullfrog", "tree frog", "tailed frog", "loggerhead sea turtle", "leatherback sea turtle", "mud turtle", "terrapin", "box turtle", "banded gecko", "green iguana", "Carolina anole", "desert grassland whiptail lizard", "agama", "frilled-necked lizard", "alligator lizard", "Gila monster", "European green lizard", "chameleon", "Komodo dragon", "Nile crocodile", "American alligator", "triceratops", "worm snake", "ring-necked snake", "eastern hog-nosed snake", "smooth green snake", "kingsnake", "garter snake", "water snake", "vine snake", "night snake", "boa constrictor", "African rock python", "Indian cobra", "green mamba", "sea snake", "Saharan horned viper", "eastern diamondback rattlesnake", "sidewinder rattlesnake", "trilobite", "harvestman", "scorpion", "yellow garden spider", "barn spider", "European garden spider", "southern black widow", "tarantula", "wolf spider", "tick", "centipede", "black grouse", "ptarmigan", "ruffed grouse", "prairie grouse", "peafowl", "quail", "partridge", "african grey parrot", "macaw", "sulphur-crested cockatoo", "lorikeet", "coucal", "bee eater", "hornbill", "hummingbird", "jacamar", "toucan", "duck", "red-breasted merganser", "goose", "black swan", "tusker", "echidna", "platypus", "wallaby", "koala", "wombat", "jellyfish", "sea anemone", "brain coral", "flatworm", "nematode", "conch", "snail", "slug", "sea slug", "chiton", "chambered nautilus", "Dungeness crab", "rock crab", "fiddler crab", "red king crab", "American lobster", "spiny lobster", "crayfish", "hermit crab", "isopod", "white stork", "black stork", "spoonbill", "flamingo", "little blue heron", "great egret", "bittern bird", "crane bird", "limpkin", "common gallinule", "American coot", "bustard", "ruddy turnstone", "dunlin", "common redshank", "dowitcher", "oystercatcher", "pelican", "king penguin", "albatross", "grey whale", "killer whale", "dugong", "sea lion", "Chihuahua", "Japanese Chin", "Maltese", "Pekingese", "Shih Tzu", "King Charles Spaniel", "Papillon", "toy terrier", "Rhodesian Ridgeback", "Afghan Hound", "Basset Hound", "Beagle", "Bloodhound", "Bluetick Coonhound", "Black and Tan Coonhound", "Treeing Walker Coonhound", "English foxhound", "Redbone Coonhound", "borzoi", "Irish Wolfhound", "Italian Greyhound", "Whippet", "Ibizan Hound", "Norwegian Elkhound", "Otterhound", "Saluki", "Scottish Deerhound", "Weimaraner", "Staffordshire Bull Terrier", "American Staffordshire Terrier", "Bedlington Terrier", "Border Terrier", "Kerry Blue Terrier", "Irish Terrier", "Norfolk Terrier", "Norwich Terrier", "Yorkshire Terrier", "Wire Fox Terrier", "Lakeland Terrier", "Sealyham Terrier", "Airedale Terrier", "Cairn Terrier", "Australian Terrier", "Dandie Dinmont Terrier", "Boston Terrier", "Miniature Schnauzer", "Giant Schnauzer", "Standard Schnauzer", "Scottish Terrier", "Tibetan Terrier", "Australian Silky Terrier", "Soft-coated Wheaten Terrier", "West Highland White Terrier", "Lhasa Apso", "Flat-Coated Retriever", "Curly-coated Retriever", "Golden Retriever", "Labrador Retriever", "Chesapeake Bay Retriever", "German Shorthaired Pointer", "Vizsla", "English Setter", "Irish Setter", "Gordon Setter", "Brittany dog", "Clumber Spaniel", "English Springer Spaniel", "Welsh Springer Spaniel", "Cocker Spaniel", "Sussex Spaniel", "Irish Water Spaniel", "Kuvasz", "Schipperke", "Groenendael dog", "Malinois", "Briard", "Australian Kelpie", "Komondor", "Old English Sheepdog", "Shetland Sheepdog", "collie", "Border Collie", "Bouvier des Flandres dog", "Rottweiler", "German Shepherd Dog", "Dobermann", "Miniature Pinscher", "Greater Swiss Mountain Dog", "Bernese Mountain Dog", "Appenzeller Sennenhund", "Entlebucher Sennenhund", "Boxer", "Bullmastiff", "Tibetan Mastiff", "French Bulldog", "Great Dane", "St. Bernard", "husky", "Alaskan Malamute", "Siberian Husky", "Dalmatian", "Affenpinscher", "Basenji", "pug", "Leonberger", "Newfoundland dog", "Great Pyrenees dog", "Samoyed", "Pomeranian", "Chow Chow", "Keeshond", "brussels griffon", "Pembroke Welsh Corgi", "Cardigan Welsh Corgi", "Toy Poodle", "Miniature Poodle", "Standard Poodle", "Mexican hairless dog (xoloitzcuintli)", "grey wolf", "Alaskan tundra wolf", "red wolf or maned wolf", "coyote", "dingo", "dhole", "African wild dog", "hyena", "red fox", "kit fox", "Arctic fox", "grey fox", "tabby cat", "tiger cat", "Persian cat", "Siamese cat", "Egyptian Mau", "cougar", "lynx", "leopard", "snow leopard", "jaguar", "lion", "tiger", "cheetah", "brown bear", "American black bear", "polar bear", "sloth bear", "mongoose", "meerkat", "tiger beetle", "ladybug", "ground beetle", "longhorn beetle", "leaf beetle", "dung beetle", "rhinoceros beetle", "weevil", "fly", "bee", "ant", "grasshopper", "cricket insect", "stick insect", "cockroach", "praying mantis", "cicada", "leafhopper", "lacewing", "dragonfly", "damselfly", "red admiral butterfly", "ringlet butterfly", "monarch butterfly", "small white butterfly", "sulphur butterfly", "gossamer-winged butterfly", "starfish", "sea urchin", "sea cucumber", "cottontail rabbit", "hare", "Angora rabbit", "hamster", "porcupine", "fox squirrel", "marmot", "beaver", "guinea pig", "common sorrel horse", "zebra", "pig", "wild boar", "warthog", "hippopotamus", "ox", "water buffalo", "bison", "ram (adult male sheep)", "bighorn sheep", "Alpine ibex", "hartebeest", "impala (antelope)", "gazelle", "arabian camel", "llama", "weasel", "mink", "European polecat", "black-footed ferret", "otter", "skunk", "badger", "armadillo", "three-toed sloth", "orangutan", "gorilla", "chimpanzee", "gibbon", "siamang", "guenon", "patas monkey", "baboon", "macaque", "langur", "black-and-white colobus", "proboscis monkey", "marmoset", "white-headed capuchin", "howler monkey", "titi monkey", "Geoffroy's spider monkey", "common squirrel monkey", "ring-tailed lemur", "indri", "Asian elephant", "African bush elephant", "red panda", "giant panda", "snoek fish", "eel", "silver salmon", "rock beauty fish", "clownfish", "sturgeon", "gar fish", "lionfish", "pufferfish", "abacus", "abaya", "academic gown", "accordion", "acoustic guitar", "aircraft carrier", "airliner", "airship", "altar", "ambulance", "amphibious vehicle", "analog clock", "apiary", "apron", "trash can", "assault rifle", "backpack", "bakery", "balance beam", "balloon", "ballpoint pen", "Band-Aid", "banjo", "baluster / handrail", "barbell", "barber chair", "barbershop", "barn", "barometer", "barrel", "wheelbarrow", "baseball", "basketball", "bassinet", "bassoon", "swimming cap", "bath towel", "bathtub", "station wagon", "lighthouse", "beaker", "military hat (bearskin or shako)", "beer bottle", "beer glass", "bell tower", "baby bib", "tandem bicycle", "bikini", "ring binder", "binoculars", "birdhouse", "boathouse", "bobsleigh", "bolo tie", "poke bonnet", "bookcase", "bookstore", "bottle cap", "hunting bow", "bow tie", "brass memorial plaque", "bra", "breakwater", "breastplate", "broom", "bucket", "buckle", "bulletproof vest", "high-speed train", "butcher shop", "taxicab", "cauldron", "candle", "cannon", "canoe", "can opener", "cardigan", "car mirror", "carousel", "tool kit", "cardboard box / carton", "car wheel", "automated teller machine", "cassette", "cassette player", "castle", "catamaran", "CD player", "cello", "mobile phone", "chain", "chain-link fence", "chain mail", "chainsaw", "storage chest", "chiffonier", "bell or wind chime", "china cabinet", "Christmas stocking", "church", "movie theater", "cleaver", "cliff dwelling", "cloak", "clogs", "cocktail shaker", "coffee mug", "coffeemaker", "spiral or coil", "combination lock", "computer keyboard", "candy store", "container ship", "convertible", "corkscrew", "cornet", "cowboy boot", "cowboy hat", "cradle", "construction crane", "crash helmet", "crate", "infant bed", "Crock Pot", "croquet ball", "crutch", "cuirass", "dam", "desk", "desktop computer", "rotary dial telephone", "diaper", "digital clock", "digital watch", "dining table", "dishcloth", "dishwasher", "disc brake", "dock", "dog sled", "dome", "doormat", "drilling rig", "drum", "drumstick", "dumbbell", "Dutch oven", "electric fan", "electric guitar", "electric locomotive", "entertainment center", "envelope", "espresso machine", "face powder", "feather boa", "filing cabinet", "fireboat", "fire truck", "fire screen", "flagpole", "flute", "folding chair", "football helmet", "forklift", "fountain", "fountain pen", "four-poster bed", "freight car", "French horn", "frying pan", "fur coat", "garbage truck", "gas mask or respirator", "gas pump", "goblet", "go-kart", "golf ball", "golf cart", "gondola", "gong", "gown", "grand piano", "greenhouse", "radiator grille", "grocery store", "guillotine", "hair clip", "hair spray", "half-track", "hammer", "hamper", "hair dryer", "hand-held computer", "handkerchief", "hard disk drive", "harmonica", "harp", "combine harvester", "hatchet", "holster", "home theater", "honeycomb", "hook", "hoop skirt", "gymnastic horizontal bar", "horse-drawn vehicle", "hourglass", "iPod", "clothes iron", "carved pumpkin", "jeans", "jeep", "T-shirt", "jigsaw puzzle", "rickshaw", "joystick", "kimono", "knee pad", "knot", "lab coat", "ladle", "lampshade", "laptop computer", "lawn mower", "lens cap", "letter opener", "library", "lifeboat", "lighter", "limousine", "ocean liner", "lipstick", "slip-on shoe", "lotion", "music speaker", "loupe magnifying glass", "sawmill", "magnetic compass", "messenger bag", "mailbox", "tights", "one-piece bathing suit", "manhole cover", "maraca", "marimba", "mask", "matchstick", "maypole", "maze", "measuring cup", "medicine cabinet", "megalith", "microphone", "microwave oven", "military uniform", "milk can", "minibus", "miniskirt", "minivan", "missile", "mitten", "mixing bowl", "mobile home", "ford model t", "modem", "monastery", "monitor", "moped", "mortar and pestle", "graduation cap", "mosque", "mosquito net", "vespa", "mountain bike", "tent", "computer mouse", "mousetrap", "moving van", "muzzle", "metal nail", "neck brace", "necklace", "baby pacifier", "notebook computer", "obelisk", "oboe", "ocarina", "odometer", "oil filter", "pipe organ", "oscilloscope", "overskirt", "bullock cart", "oxygen mask", "product packet / packaging", "paddle", "paddle wheel", "padlock", "paintbrush", "pajamas", "palace", "pan flute", "paper towel", "parachute", "parallel bars", "park bench", "parking meter", "railroad car", "patio", "payphone", "pedestal", "pencil case", "pencil sharpener", "perfume", "Petri dish", "photocopier", "plectrum", "Pickelhaube", "picket fence", "pickup truck", "pier", "piggy bank", "pill bottle", "pillow", "ping-pong ball", "pinwheel", "pirate ship", "drink pitcher", "block plane", "planetarium", "plastic bag", "plate rack", "farm plow", "plunger", "Polaroid camera", "pole", "police van", "poncho", "pool table", "soda bottle", "plant pot", "potter's wheel", "power drill", "prayer rug", "printer", "prison", "missile", "projector", "hockey puck", "punching bag", "purse", "quill", "quilt", "race car", "racket", "radiator", "radio", "radio telescope", "rain barrel", "recreational vehicle", "fishing casting reel", "reflex camera", "refrigerator", "remote control", "restaurant", "revolver", "rifle", "rocking chair", "rotisserie", "eraser", "rugby ball", "ruler measuring stick", "sneaker", "safe", "safety pin", "salt shaker", "sandal", "sarong", "saxophone", "scabbard", "weighing scale", "school bus", "schooner", "scoreboard", "CRT monitor", "screw", "screwdriver", "seat belt", "sewing machine", "shield", "shoe store", "shoji screen / room divider", "shopping basket", "shopping cart", "shovel", "shower cap", "shower curtain", "ski", "balaclava ski mask", "sleeping bag", "slide rule", "sliding door", "slot machine", "snorkel", "snowmobile", "snowplow", "soap dispenser", "soccer ball", "sock", "solar thermal collector", "sombrero", "soup bowl", "keyboard space bar", "space heater", "space shuttle", "spatula", "motorboat", "spider web", "spindle", "sports car", "spotlight", "stage", "steam locomotive", "through arch bridge", "steel drum", "stethoscope", "scarf", "stone wall", "stopwatch", "stove", "strainer", "tram", "stretcher", "couch", "stupa", "submarine", "suit", "sundial", "sunglasses", "sunglasses", "sunscreen", "suspension bridge", "mop", "sweatshirt", "swim trunks / shorts", "swing", "electrical switch", "syringe", "table lamp", "tank", "tape player", "teapot", "teddy bear", "television", "tennis ball", "thatched roof", "front curtain", "thimble", "threshing machine", "throne", "tile roof", "toaster", "tobacco shop", "toilet seat", "torch", "totem pole", "tow truck", "toy store", "tractor", "semi-trailer truck", "tray", "trench coat", "tricycle", "trimaran", "tripod", "triumphal arch", "trolleybus", "trombone", "hot tub", "turnstile", "typewriter keyboard", "umbrella", "unicycle", "upright piano", "vacuum cleaner", "vase", "vaulted or arched ceiling", "velvet fabric", "vending machine", "vestment", "viaduct", "violin", "volleyball", "waffle iron", "wall clock", "wallet", "wardrobe", "military aircraft", "sink", "washing machine", "water bottle", "water jug", "water tower", "whiskey jug", "whistle", "hair wig", "window screen", "window shade", "Windsor tie", "wine bottle", "airplane wing", "wok", "wooden spoon", "wool", "split-rail fence", "shipwreck", "sailboat", "yurt", "website", "comic book", "crossword", "traffic or street sign", "traffic light", "dust jacket", "menu", "plate", "guacamole", "consomme", "hot pot", "trifle", "ice cream", "popsicle", "baguette", "bagel", "pretzel", "cheeseburger", "hot dog", "mashed potatoes", "cabbage", "broccoli", "cauliflower", "zucchini", "spaghetti squash", "acorn squash", "butternut squash", "cucumber", "artichoke", "bell pepper", "cardoon", "mushroom", "Granny Smith apple", "strawberry", "orange", "lemon", "fig", "pineapple", "banana", "jackfruit", "cherimoya (custard apple)", "pomegranate", "hay", "carbonara", "chocolate syrup", "dough", "meatloaf", "pizza", "pot pie", "burrito", "red wine", "espresso", "tea cup", "eggnog", "mountain", "bubble", "cliff", "coral reef", "geyser", "lakeshore", "promontory", "sandbar", "beach", "valley", "volcano", "baseball player", "bridegroom", "scuba diver", "rapeseed", "daisy", "yellow lady's slipper", "corn", "acorn", "rose hip", "horse chestnut seed", "coral fungus", "agaric", "gyromitra", "stinkhorn mushroom", "earth star fungus", "hen of the woods mushroom", "bolete", "corn cob", "toilet paper"]

A subset of these class names are modified from the default ImageNet class names sourced from Anish Athalye's imagenet-simple-labels.

These edits were made via trial and error and concentrated on the lowest performing classes according to top_1 and top_5 accuracy on the ImageNet training set for the RN50, RN101, and RN50x4 models. These tweaks improve top_1 by 1.5% on ViT-B/32 over using the default class names. Alec got bored somewhere along the way as gains started to diminish and never finished updating / tweaking the list. He also didn't revisit this with the better performing RN50x16, RN50x64, or any of the ViT models. He thinks it's likely another 0.5% to 1% top_1 could be gained from further work here. It'd be interesting to more rigorously study / understand this.

Some examples beyond the crane/crane -> construction crane / bird crane issue mentioned in Section 3.1.4 of the paper include:

- CLIP interprets "nail" as "fingernail" so we changed the label to "metal nail".
- ImageNet kite class refers to the bird of prey, not the flying toy, so we changed "kite" to "kite (bird of prey)"
- The ImageNet class for red wolf seems to include a lot of mislabeled maned wolfs so we changed "red wolf" to "red wolf or maned wolf"

In [7]:
imagenet_templates = [
    'a bad photo of a {}.',
    'a photo of many {}.',
    'a sculpture of a {}.',
    'a photo of the hard to see {}.',
    'a low resolution photo of the {}.',
    'a rendering of a {}.',
    'graffiti of a {}.',
    'a bad photo of the {}.',
    'a cropped photo of the {}.',
    'a tattoo of a {}.',
    'the embroidered {}.',
    'a photo of a hard to see {}.',
    'a bright photo of a {}.',
    'a photo of a clean {}.',
    'a photo of a dirty {}.',
    'a dark photo of the {}.',
    'a drawing of a {}.',
    'a photo of my {}.',
    'the plastic {}.',
    'a photo of the cool {}.',
    'a close-up photo of a {}.',
    'a black and white photo of the {}.',
    'a painting of the {}.',
    'a painting of a {}.',
    'a pixelated photo of the {}.',
    'a sculpture of the {}.',
    'a bright photo of the {}.',
    'a cropped photo of a {}.',
    'a plastic {}.',
    'a photo of the dirty {}.',
    'a jpeg corrupted photo of a {}.',
    'a blurry photo of the {}.',
    'a photo of the {}.',
    'a good photo of the {}.',
    'a rendering of the {}.',
    'a {} in a video game.',
    'a photo of one {}.',
    'a doodle of a {}.',
    'a close-up photo of the {}.',
    'a photo of a {}.',
    'the origami {}.',
    'the {} in a video game.',
    'a sketch of a {}.',
    'a doodle of the {}.',
    'a origami {}.',
    'a low resolution photo of a {}.',
    'the toy {}.',
    'a rendition of the {}.',
    'a photo of the clean {}.',
    'a photo of a large {}.',
    'a rendition of a {}.',
    'a photo of a nice {}.',
    'a photo of a weird {}.',
    'a blurry photo of a {}.',
    'a cartoon {}.',
    'art of a {}.',
    'a sketch of the {}.',
    'a embroidered {}.',
    'a pixelated photo of a {}.',
    'itap of the {}.',
    'a jpeg corrupted photo of the {}.',
    'a good photo of a {}.',
    'a plushie {}.',
    'a photo of the nice {}.',
    'a photo of the small {}.',
    'a photo of the weird {}.',
    'the cartoon {}.',
    'art of the {}.',
    'a drawing of the {}.',
    'a photo of the large {}.',
    'a black and white photo of a {}.',
    'the plushie {}.',
    'a dark photo of a {}.',
    'itap of a {}.',
    'graffiti of the {}.',
    'a toy {}.',
    'itap of my {}.',
    'a photo of a cool {}.',
    'a photo of a small {}.',
    'a tattoo of the {}.',
]

print(f"{len(imagenet_classes)} classes, {len(imagenet_templates)} templates")

1000 classes, 80 templates


A similar, intuition-guided trial and error based on the ImageNet training set was used for templates. This list is pretty haphazard and was gradually made / expanded over the course of about a year of the project and was revisited / tweaked every few months. A surprising / weird thing was adding templates intended to help ImageNet-R performance (specifying different possible renditions of an object) improved standard ImageNet accuracy too.

After the 80 templates were "locked" for the paper, we ran sequential forward selection over the list of 80 templates. The search terminated after ensembling 7 templates and selected them in the order below.

1. itap of a {}.
2. a bad photo of the {}.
3. a origami {}.
4. a photo of the large {}.
5. a {} in a video game.
6. art of the {}.
7. a photo of the small {}.

Speculating, we think it's interesting to see different scales (large and small), a difficult view (a bad photo), and "abstract" versions (origami, video game, art), were all selected for, but we haven't studied this in any detail. This subset performs a bit better than the full 80 ensemble reported in the paper, especially for the smaller models.

# Loading the Images【加载数据】

The ILSVRC2012 datasets are no longer available for download publicly. We instead download the ImageNet-V2 dataset by [Recht et al.](https://arxiv.org/abs/1902.10811).

If you have the ImageNet dataset downloaded, you can replace the dataset with the official torchvision loader, e.g.:

```python
images = torchvision.datasets.ImageNet("path/to/imagenet", split='val', transform=preprocess)
```

In [8]:
# 使用pip安装ImageNetV2数据集库
# ! 表示在Jupyter Notebook中执行shell命令
# pip install: Python包安装命令
# git+https://...: 从GitHub仓库安装包
# modestyachts/ImageNetV2_pytorch: 包含ImageNetV2数据集的PyTorch实现
# ImageNetV2是一个用于评估ImageNet模型鲁棒性的数据集
! pip install git+https://github.com/modestyachts/ImageNetV2_pytorch

# 从安装的包中导入ImageNetV2Dataset类
# 这个类提供了对ImageNetV2数据集的封装
from imagenetv2_pytorch import ImageNetV2Dataset

# 创建ImageNetV2数据集实例
# ImageNetV2Dataset(): 初始化数据集
# transform=preprocess: 使用之前定义的CLIP预处理函数
#   preprocess包含: 调整大小、中心裁剪、转换为张量、归一化
#   将ImageNetV2图像转换为CLIP需要的224x224格式
# images: 数据集对象，可以像列表一样索引访问
images = ImageNetV2Dataset(transform=preprocess)

# 创建数据加载器，用于批量加载数据
# torch.utils.data.DataLoader(): PyTorch数据加载器
# 参数1: images - 要加载的数据集对象
# 参数2: batch_size=32 - 每个批次加载32个样本
# 参数3: num_workers=2 - 使用2个子进程加载数据（加速数据读取）
# loader: 数据加载器对象，可以迭代获取批次数据
loader = torch.utils.data.DataLoader(images, batch_size=32, num_workers=2)

  Cloning https://github.com/modestyachts/ImageNetV2_pytorch to /tmp/pip-req-build-0kih0kn2
  Running command git clone -q https://github.com/modestyachts/ImageNetV2_pytorch /tmp/pip-req-build-0kih0kn2
  Created wheel for imagenetv2-pytorch: filename=imagenetv2_pytorch-0.1-py3-none-any.whl size=2663 sha256=ac31e0ed9c61afc5e0271eed315d3a82844a79ae64f8ce43fc1c98928cec129f
  Stored in directory: /tmp/pip-ephem-wheel-cache-745b5n1m/wheels/ab/ee/f4/73bce5c7f68d28ce632ef33ae87ce60aaca021eb2b3b31a6fa
Successfully built imagenetv2-pytorch
Dataset matched-frequency not found on disk, downloading....


100%|██████████| 1.26G/1.26G [01:02<00:00, 20.2MiB/s]


Extracting....


# Creating zero-shot classifier weights【1000个类别文本特征】

In [9]:
# 定义零样本分类器函数
# 功能：为每个类别生成文本特征（嵌入向量），用作分类权重
# 这是CLIP零样本分类的核心：将类别名称转换为模型可理解的文本特征
def zeroshot_classifier(classnames, templates):
    # 禁用梯度计算，只进行前向传播（推理模式）
    with torch.no_grad():
        # 初始化空列表，用于存储每个类别的特征向量
        zeroshot_weights = []

        # 遍历所有类别名称，使用tqdm显示进度条
        # classnames: 类别名称列表，如["cat", "dog", "car", ...]
        # tqdm(): 显示循环进度，便于监控处理大量类别时的进度
        for classname in tqdm(classnames):
            # 为当前类别生成多个文本描述
            # templates: 文本模板列表，如["a photo of a {}", "a picture of a {}", ...]
            # template.format(classname): 将类别名插入模板
            # 示例: 模板"a photo of a {}" + 类别"cat" → "a photo of a cat"
            texts = [template.format(classname) for template in templates]

            # 将文本描述转换为token
            # clip.tokenize(texts): 将文本列表转换为token ID张量
            # .cuda(): 移动到GPU以加速计算
            # 输出形状: [len(templates), 77] (模板数量, 最大文本长度)
            texts = clip.tokenize(texts).cuda()

            # 提取文本特征（嵌入）
            # model.encode_text(texts): 使用CLIP文本编码器提取特征
            # 输出形状: [len(templates), 512] (模板数量, 特征维度)
            class_embeddings = model.encode_text(texts)

            # 对每个模板的特征进行L2归一化
            # class_embeddings.norm(dim=-1, keepdim=True): 计算每个512维向量的范数
            # /= : 原地除法，使每个特征向量成为单位向量
            # 归一化后，点积 = 余弦相似度
            class_embeddings /= class_embeddings.norm(dim=-1, keepdim=True)

            # 计算多个模板特征的平均值
            # .mean(dim=0): 沿模板维度求平均
            # 形状变化: [len(templates), 512] → [512]
            # 目的: 融合不同模板的信息，得到更稳健的类别特征
            class_embedding = class_embeddings.mean(dim=0)

            # 对平均后的特征再次归一化
            # class_embedding.norm(): 计算512维向量的范数
            # /= : 使最终类别特征成为单位向量
            class_embedding /= class_embedding.norm()

            # 将当前类别的特征向量添加到列表
            zeroshot_weights.append(class_embedding)

        # 将所有类别的特征向量堆叠成一个矩阵【dim=0更好理解，=1是一列一列的，方便后面算相似度矩阵】
        # torch.stack(zeroshot_weights, dim=1):
        #   将列表中的向量沿第1维度堆叠
        #   每个zeroshot_weights[i]形状为[512]
        #   堆叠后形状: [512, num_classes]
        # .cuda(): 确保矩阵在GPU上
        zeroshot_weights = torch.stack(zeroshot_weights, dim=1).cuda()

    # 返回零样本分类权重矩阵
    # 形状: [512, num_classes]
    #   512: CLIP特征空间维度
    #   num_classes: 类别数量
    # 这个矩阵将用于计算图像与所有类别的相似度
    return zeroshot_weights


# 调用函数生成ImageNet的零样本分类权重
# imagenet_classes: ImageNet的1000个类别名称列表
# imagenet_templates: 文本模板列表
# zeroshot_weights: 生成的分类权重矩阵，形状[512, 1000]
zeroshot_weights = zeroshot_classifier(imagenet_classes, imagenet_templates)

# Zero-shot prediction【零样本ImageNet数据集预测分类性能】

In [10]:
# 定义准确率计算函数
# 功能：计算分类任务中top-k准确率
# 在ImageNet等大型分类任务中，常用top-1和top-5准确率
def accuracy(output, target, topk=(1,)):
    """
    参数:
    output: 模型输出logits，形状为[batch_size, num_classes]
            batch_size: 批次大小
            num_classes: 类别数量
            例如: [32, 1000] 表示32个样本，1000个类别

    target: 真实标签，形状为[batch_size]
            每个元素是0到num_classes-1的整数
            例如: [32] 表示32个样本的真实类别索引

    topk: 元组，指定要计算哪些top-k准确率
          例如: (1, 5) 表示计算top-1和top-5准确率
          默认只计算top-1

    返回:
    列表，包含每个k值的正确预测数量（不是百分比）
    例如: [15, 28] 表示top-1正确15个，top-5正确28个【在一个批次下，有多少个预测正确了】【top-5：预测的前5个类别只要有真正的标签都算对】
    """

    # 步骤1: 获取预测的top-k类别索引
    # output.topk(max(topk), 1, True, True): 找到每个样本的top-k预测
    #   max(topk): 需要的最大的k值，如topk=(1,5)则max(topk)=5
    #   1: 在维度1（类别维度）上查找
    #   True: 按降序排列（值最大的在前）
    #   True: 返回排序后的结果
    #   [1]: topk返回(值, 索引)，我们只需要索引
    #   .t(): 转置操作，将形状从[batch_size, k]转为[k, batch_size]
    #
    # 示例（简化）:
    #   假设output形状[2, 5]（2个样本，5个类别）:
    #   output = [[0.1, 0.9, 0.3, 0.2, 0.5],  # 样本0
    #             [0.4, 0.2, 0.8, 0.1, 0.3]]  # 样本1
    #   pred = output.topk(3, 1, True, True)[1]
    #   pred = [[1, 4, 2],  # 样本0: 最可能是类别1, 然后是4, 然后是2
    #           [2, 0, 4]]  # 样本1: 最可能是类别2, 然后是0, 然后是4
    #   pred.t() = [[1, 2],  # 所有样本的top-1预测
    #               [4, 0],  # 所有样本的top-2预测
    #               [2, 4]]  # 所有样本的top-3预测
    pred = output.topk(max(topk), 1, True, True)[1].t()

    # 步骤2: 检查预测是否正确
    # target.view(1, -1): 将target从[batch_size]变为[1, batch_size]
    #   view(1, -1): -1表示自动计算该维度大小
    #   example: target=[0, 2] → [[0, 2]]
    #
    # .expand_as(pred): 扩展为与pred相同的形状
    #   pred形状: [k, batch_size]
    #   扩展后: 从[1, batch_size]扩展为[k, batch_size]
    #   每个k值都复制相同的真实标签
    #   example: [[0, 2]]扩展为[[0, 2], [0, 2], [0, 2]] (k=3时)
    #
    # pred.eq(...): 逐元素比较是否相等
    #   返回布尔张量，形状与pred相同
    #   example: pred=[[1,2], [4,0], [2,4]], 扩展后的target=[[0,2], [0,2], [0,2]]
    #   比较结果: [[False, True],  # 样本0: 1≠0❌, 样本1: 2=2✓
    #              [False, False], # 样本0: 4≠0❌, 样本1: 0≠2❌
    #              [False, False]] # 样本0: 2≠0❌, 样本1: 4≠2❌
    correct = pred.eq(target.view(1, -1).expand_as(pred))

    # 步骤3: 计算每个k值的正确数量
    # 列表推导式，为topk中的每个k值计算
    return [float(correct[:k].reshape(-1).float().sum(0, keepdim=True).cpu().numpy()) for k in topk]
    # 分解步骤（对于某个k值，如k=1）:
    # 1. correct[:k]: 取前k行，形状变为[k, batch_size]
    #    k=1时: 取第一行，形状[1, batch_size]
    #    对应top-1预测的正确性
    #
    # 2. .reshape(-1): 展平为一维张量
    #    形状: [k * batch_size]
    #    k=1, batch_size=32时: 形状[32]
    #
    # 3. .float(): 布尔值转换为浮点数（True→1.0, False→0.0）
    #
    # 4. .sum(0, keepdim=True): 求和，得到正确预测的数量
    #    0: 在维度0上求和（展平后只有一维）
    #    keepdim=True: 保持维度，形状从[]变为[1]
    #
    # 5. .cpu().numpy(): 从GPU移动到CPU，转换为NumPy标量
    #
    # 6. float(...): 转换为Python浮点数
    #
    # 对于k=5: 取前5行，检查真实标签是否在前5个预测中

In [11]:
# 禁用梯度计算，进入推理模式
# 在评估阶段不需要计算梯度，节省内存和计算资源
with torch.no_grad():
    # 初始化累加器
    # top1: 累加top-1正确预测数量（浮点数）
    # top5: 累加top-5正确预测数量（浮点数）
    # n: 累加已处理的样本总数
    top1, top5, n = 0., 0., 0.

    # 遍历数据加载器中的所有批次
    # enumerate(loader): 获取批次索引和批次数据
    # tqdm(): 显示进度条，便于监控评估进度
    # i: 批次索引（0, 1, 2, ...）
    # (images, target): 批次数据，包含图像和标签
    for i, (images, target) in enumerate(tqdm(loader)):
        # 将图像数据移动到GPU
        # images: 形状为[batch_size, 3, 224, 224]的张量
        # .cuda(): 将数据从CPU移动到GPU
        images = images.cuda()

        # 将标签数据移动到GPU
        # target: 形状为[batch_size]的整数张量，表示真实类别索引
        target = target.cuda()

        # --- 预测阶段 ---
        # 提取图像特征
        # model.encode_image(images): 使用CLIP图像编码器提取特征
        # 输入: [batch_size, 3, 224, 224]
        # 输出: [batch_size, 512] (CLIP特征维度)
        image_features = model.encode_image(images)

        # 对图像特征进行L2归一化
        # image_features.norm(dim=-1, keepdim=True): 计算每个512维向量的范数
        # /= : 原地除法，使每个特征向量成为单位向量
        # 归一化后，点积 = 余弦相似度
        image_features /= image_features.norm(dim=-1, keepdim=True)

        # 计算logits（未归一化的分类分数）
        # zeroshot_weights: 形状[512, num_classes]的分类权重矩阵
        #   通过zeroshot_classifier函数预先计算
        #   每列是一个类别的文本特征向量
        # image_features @ zeroshot_weights: 矩阵乘法计算相似度
        #   形状: [batch_size, 512] × [512, num_classes] = [batch_size, num_classes]
        # 100. * : 温度缩放，相当于温度参数τ=0.01
        #   使softmax输出更"尖锐"，区分度更高
        logits = 100. * image_features @ zeroshot_weights

        # --- 准确率计算 ---
        # 调用accuracy函数计算top-1和top-5准确率
        # logits: 形状[batch_size, num_classes]的预测分数
        # target: 形状[batch_size]的真实标签
        # topk=(1, 5): 计算top-1和top-5准确率
        # acc1: 当前批次的top-1正确预测数量
        # acc5: 当前批次的top-5正确预测数量
        acc1, acc5 = accuracy(logits, target, topk=(1, 5))

        # 累加正确预测数量
        # top1 += acc1: 累加top-1正确数
        # top5 += acc5: 累加top-5正确数
        top1 += acc1
        top5 += acc5

        # 累加已处理的样本数
        # images.size(0): 当前批次的样本数（batch_size）
        # 注意：最后一个批次可能小于batch_size
        n += images.size(0)

# 计算最终的准确率百分比
# top1 / n: top-1正确率（小数形式）
# * 100: 转换为百分比
top1 = (top1 / n) * 100
top5 = (top5 / n) * 100

# 打印评估结果
# :.2f: 格式化输出，保留2位小数
print(f"Top-1 accuracy: {top1:.2f}")
print(f"Top-5 accuracy: {top5:.2f}")


Top-1 accuracy: 55.93
Top-5 accuracy: 83.36
